# 02. Review 수정 전파와 bottom-up merge

## 학습 목표

- lower layer 수정이 왜 모든 upstack branch에 전파돼야 하는지 확인합니다.
- merge 가능한 contiguous group 조건을 구현합니다.
- 일부 merge 뒤 remaining stack의 base가 어떻게 바뀌는지 관찰합니다.

실제 rebase SHA를 만드는 대신 commit label로 동작을 시뮬레이션합니다.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class PullRequestLayer:
    branch: str
    base: str
    own_commits: list[str]
    inherited_commits: list[str] = field(default_factory=list)
    approved: bool = False
    checks_passed: bool = False

    @property
    def full_history(self) -> list[str]:
        return self.inherited_commits + self.own_commits

stack = [
    PullRequestLayer("auth", "main", ["A:model"]),
    PullRequestLayer("api", "auth", ["B:endpoint"]),
    PullRequestLayer("ui", "api", ["C:login-ui"]),
]

In [ ]:
def cascade_rebase(stack: list[PullRequestLayer], start: int = 0) -> None:
    """bottom에서 top 방향으로 inherited history를 다시 구성합니다."""
    for index in range(max(0, start), len(stack)):
        if index == 0:
            stack[index].inherited_commits = []
            stack[index].base = "main"
        else:
            stack[index].base = stack[index - 1].branch
            stack[index].inherited_commits = stack[index - 1].full_history.copy()

cascade_rebase(stack)
for layer in stack:
    print(layer.branch, "base=", layer.base, "history=", layer.full_history)

In [ ]:
# bottom review에서 schema 수정 요청이 왔다고 가정합니다.
stack[0].own_commits.append("A2:add-index")
cascade_rebase(stack, start=1)

assert "A2:add-index" in stack[1].inherited_commits
assert "A2:add-index" in stack[2].inherited_commits
assert "A2:add-index" not in stack[1].own_commits

for layer in stack:
    print(layer.branch, layer.full_history)

print("수정은 auth layer 소유이며 api/ui는 rebase로 상속했습니다.")

In [ ]:
def merge_through(stack: list[PullRequestLayer], selected_index: int, main_history: list[str]) -> list[PullRequestLayer]:
    """lowest unmerged부터 selected layer까지 연속 group을 merge합니다."""
    if not 0 <= selected_index < len(stack):
        raise IndexError("selected layer가 stack 범위를 벗어났습니다")
    group = stack[:selected_index + 1]
    blocked = [layer.branch for layer in group if not (layer.approved and layer.checks_passed)]
    if blocked:
        raise ValueError(f"merge requirement 미충족: {blocked}")
    for layer in group:
        main_history.extend(layer.own_commits)
    remaining = stack[selected_index + 1:]
    if remaining:
        # GitHub가 다음 unmerged layer를 trunk로 retarget/rebase하는 동작을 흉내 냅니다.
        for layer in remaining:
            layer.inherited_commits = []
        cascade_rebase(remaining)
    return remaining

main_history: list[str] = []
for layer in stack[:2]:
    layer.approved = True
    layer.checks_passed = True

remaining = merge_through(stack, selected_index=1, main_history=main_history)
print("main history:", main_history)
print("remaining:", [(layer.branch, layer.base, layer.full_history) for layer in remaining])

assert main_history == ["A:model", "A2:add-index", "B:endpoint"]
assert remaining[0].branch == "ui" and remaining[0].base == "main"

## 확장 과제

1. mid-stack layer를 선택할 때 아래 layer가 자동 포함됨을 표시합니다.
2. lower check 실패 시 그 위 layer가 merge queue에서 제거되는 규칙을 추가합니다.
3. non-linear history를 탐지하고 rebase 필요 상태를 반환합니다.
4. signed commit requirement가 있을 때 server-side rebase를 금지하는 policy를 작성합니다.